#Fashion MNIST Project - CNN

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [2]:
torch.cuda.is_available()

True

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transforms

In [4]:
transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

##Train data and Test data

In [5]:
train_data = torchvision.datasets.FashionMNIST(
    root = "./data",
    train = True,
    download = True,
    transform = transforms
)
test_data = torchvision.datasets.FashionMNIST(
    root = "./data",
    train = False,
    download = True,
    transform = transforms
)

##Data Loader

In [6]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size = 32, shuffle = True)

test_loader = torch.utils.data.DataLoader(test_data, batch_size = 32, shuffle = False)

In [13]:
next(iter(train_loader))[0].shape

torch.Size([32, 1, 28, 28])

##Build the model

In [35]:
class KiyimCNN(nn.Module):
  def __init__(self):
    super(KiyimCNN, self).__init__()
    self.net = nn.Sequential(
        nn.Conv2d(in_channels = 1, out_channels=8, kernel_size=3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2), # 1-conv blok
        nn.Conv2d(in_channels = 8, out_channels=16, kernel_size=3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),


        # MLP
        nn.Flatten(),
        nn.Linear(16*7*7, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
        )
  def forward(self, x):
    return self.net(x)

In [36]:
model = KiyimCNN().to(device)

In [37]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.002)

##Training


In [38]:
epochs = 5

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for data, target in train_loader:
    data = data.to(device)
    target = target.to(device)
    output = model(data)
    loss = criterion(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
  print(f"Epoch: {epoch+1}, Loss Value: {total_loss/len(train_loader)}")

Epoch: 1, Loss Value: 0.4276935299595197
Epoch: 2, Loss Value: 0.28979474451939263
Epoch: 3, Loss Value: 0.24754995187123616
Epoch: 4, Loss Value: 0.2180396121546626
Epoch: 5, Loss Value: 0.19521297187457481


##Testing and Evaluation

In [39]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for data, target in test_loader:
    data, target = data.to(device), target.to(device)
    output = model(data) # Calculate model output
    _, predicted = torch.max(output.data, 1)
    total += target.size(0) # Increment total samples
    correct += (predicted == target).sum().item() # Count correct predictions

accuracy = 100 * correct / total
print(f'Accuracy: {accuracy}%')

Accuracy: 89.65%
